# Model-backed Player Research

A working lab notebook for persistent-session and typed-decision Klondike players. Use it to preserve hypotheses, prompt revisions, run summaries, and qualitative failure analysis before promoting stable results to the root `README.md`.

**Status:** P0 Sol and Luna baselines and the Jev P1.5 evaluation are complete and promoted to the root `README.md`; the first GitHub Copilot CLI Haiku smoke game is complete.

## Ten-Game Screening Protocol

Use [`experiments/LLM_EVALUATION.md`](../LLM_EVALUATION.md) as the canonical rubric for new LLM and configuration screens. Run 10 random games, generate the deterministic episode-analysis report, then record the confirmed rating, representative games, dominant failure mode, and next action here. These screens decide whether a model deserves a larger run; they are not promoted as precise win-rate estimates.

## Research Question

Can a current model formulate and execute a winning Klondike strategy, and how much does explicit domain guidance change its play?

### Initial hypothesis

An unguided model may produce no wins. Meaningful wins may require prompt refinement that turns model knowledge or typed choices into consistent play across a complete game.

## Experiment Contract

Keep these conditions fixed unless a run explicitly studies one of them. Typed decision APIs use the equivalent stateful contract by receiving compact observation history on every request.

- One fresh CLI or chat session per game; never reuse context across deals.
- Ask the model to state the Klondike strategy it already knows before showing the first board.
- Resume the same session for every move so strategy, boards, commands, and feedback accumulate.
- Disable engine guidance and do not send engine-recommended moves.
- Generate legal moves in the Java engine, constrain structured output when the client supports it, and always validate the selected command against that set.
- Use subscription-authenticated CLI access rather than direct provider API billing.
- Record exact model ID, reasoning effort, client and version, prompt version, random-deal method, and maximum move limit.
- Use a fresh independent random shuffle for every game; obtain confidence through sample size rather than replaying a fixed deal suite.

In [ ]:
from pathlib import Path
import json

# Adjust only when the experiment contract intentionally changes.
experiment_contract = {
    "session_mode": "persistent_per_game",
    "strategy_source": "model_authored_before_first_board",
    "engine_guidance": False,
    "access_path": "subscription_authenticated_cli",
    "max_moves": 1000,
    "confidence_level": 0.95,
}
experiment_contract

## Prompt Version Ledger

Assign a version before every prompt change. Preserve the exact text or link to the commit that contains it. Record the expected effect before running games.

| Version | Description | Expected effect | Source |
|---|---|---|---|
| P0 | Ask what Klondike rules and strategy the model already knows; provide only board notation and legal moves during play | Establish an unassisted persistent-session baseline | `GamePrompts.STRATEGY_PROMPT` |
| P1.5 | Supply corrected draw-three rules, shared decision priorities, progress requirements, and an expanded game interface | Test whether reusable domain guidance improves coherent play and win rate over P0 | `GamePrompts`; select with `-Dgame.prompt.profile=detailed` |

## Run Ledger

One row represents one fixed combination of model, reasoning effort, prompt version, and deal suite. Do not combine pilot runs with confirmatory runs unless their contracts are identical.

| Run ID | Date | Model | Reasoning | Prompt | Deals | Games | Wins | Win % and 95% CI | Avg time | Avg moves | Notes |
|---|---|---|---|---|---|---:|---:|---|---|---|---|
| `p0-sol-medium-smoke-20260924` | 2026-09-24 | `gpt-5.6-sol` | Medium | P0 | Independent random shuffle | 1 | 0 | 0.00% (n=1; interval pending) | 480.706s | 60.00 | Protocol succeeded; model quit with only potentially repetitive tableau moves remaining |
| `p0-sol-medium-100-20260925` | 2026-09-24 to 2026-09-25 | `gpt-5.6-sol` | Medium | P0 | Independent random shuffle | 100 | 18 | 18.00% ± 7.53% | 474.557s | 59.68 | Completed across 10, 1, 9, and 80-game batches; two interrupted partial games excluded |
| `p0-luna-medium-100-20260926` | 2026-09-25 to 2026-09-26 | `gpt-5.6-luna` | Medium | P0 | Independent random shuffle | 100 | 5 | 5.00% ± 4.27% | 890.632s | 130.78 | First 97 completed games plus a 3-game completion batch; one interrupted partial game excluded |
| `p0-copilot-haiku-smoke-20260927` | 2026-09-27 | `claude-haiku-4.5` via GitHub Copilot CLI | Default | P0 | Independent random shuffle | 1 | 0 | 0.00% (n=1; interval pending) | 341.722s | 44.00 | Persistent-session protocol succeeded with no corrections or transient failures; CLI reported 15.18 usage units |
| `p0-jev-stateful-smoke-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P0 | Independent random shuffle | 1 | 0 | 0.00% (n=1; interval pending) | 71.478s | 200.00 | Stateful structured observations; reached move cap; 3,238,750 input tokens |
| `p0-jev-screen-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P0 | Independent random shuffle | 10 | 0 | 0.00% (Wilson 95% interval: 0.0%-27.8%) | 68.979s | 200.00 | All games reached the move cap; 1,971 of 2,000 choices were stock turns |
| `p1-jev-stateful-smoke-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P1 | Independent random shuffle | 1 | 0 | 0.00% (n=1; interval pending) | 69.480s | 200.00 | Corrected rules and eight priorities; reached move cap; 3,171,789 input tokens |
| `p1.2-jev-ranked-smoke-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P1 / J-P1.2 | Independent random shuffle | 1 | 0 | 0.00% (n=1; interval pending) | 68.968s | 200.00 | Numeric option ranks regressed to 194 stock turns; rejected |
| `p1.3-jev-assessed-pilot-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P1 / J-P1.3 | Independent random shuffle | 3 | 0 | 0.00% (n=3; interval pending) | 68.578s | 200.00 | Natural-language simulated consequences plus compact history; two games reached 11 foundation cards |
| `p1.4-jev-stall-pilot-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P1 / J-P1.4 | Independent random shuffle | 3 | 0 | 0.00% (n=3; interval pending) | 67.486s | 200.00 | Added consecutive-turn and no-progress counters; all games made card moves |
| `p1.5-jev-progress-pilot-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P1 / J-P1.5 | Independent random shuffle | 3 | 0 | 0.00% (n=3; interval pending) | 68.540s | 200.00 | Marked immediate reveal/foundation options as required; one game reached 30 foundation cards and one hidden card |
| `p1.5-jev-progress-confirm-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P1 / J-P1.5 | Independent random shuffle | 5 | 2 | 40.00% ± 42.94% | 59.179s | 178.00 | First Jev wins, in 156 and 134 moves; small diagnostic sample, not a stable win-rate estimate |
| `p1.5-jev-detailed-100-20260928` | 2026-09-28 | `jev-1.13.0` via TypeSafe System One | System One | P1.5 | Independent random shuffle | 100 | 10 | 10.00% ± 5.88% | 65.582s | 193.52 | Combined 59- and 41-game batches; one billing-interrupted partial game excluded |

In [ ]:
# Future result runners should emit one JSON object per configuration.
# Keep generated runtime data out of git; promote only small, reviewed records.
result_candidates = [
    Path("experiments/runtime/artifacts/llm-player-results.jsonl"),
    Path("../runtime/artifacts/llm-player-results.jsonl"),
]
results_path = next((path for path in result_candidates if path.exists()), None)
records = []
if results_path:
    records = [json.loads(line) for line in results_path.read_text().splitlines() if line.strip()]
records

## Failure Taxonomy

Classify failed games so prompt changes respond to observed behavior rather than intuition alone.

- **Premature foundation move:** loses a card needed for tableau development.
- **Missed reveal:** chooses progress that does not uncover a face-down tableau card.
- **Poor empty-column use:** creates or fills an empty column without a productive King plan.
- **Stock-memory failure:** forgets previously seen waste cards or the order of a stock pass.
- **Cycle:** returns to repeated board states without new information or progress.
- **Short-horizon choice:** selects an attractive immediate move that blocks a known follow-up.
- **Premature quit:** stops while productive legal play remains.
- **Context drift:** stops following its stated strategy later in a long game.
- **Protocol failure:** returns malformed output or a command outside the legal set.
- **Move-limit termination:** reaches the configured safety cap.

## Run Notes Template

Copy this block for each meaningful run:

### Run `<run-id>`: `<short description>`

- **Question:**
- **Prediction made before running:**
- **Configuration:**
- **Result:**
- **Representative session IDs:**
- **Observed failure modes:**
- **Interpretation:**
- **Decision for the next prompt version:**

### Run `p0-sol-medium-smoke-20260924`: first persistent-session smoke game

- **Question:** Can the unassisted P0 model complete a game, and will persistent session resumption plus the response schema work for a full game?
- **Prediction made before running:** The model may not win; command formatting may fail before strategy can be evaluated.
- **Configuration:** `gpt-5.6-sol`, medium reasoning, Codex CLI 0.154.0, P0, persistent session, engine guidance disabled, independent random shuffle, 1,000-move cap.
- **Result:** 0/1 wins, 60 moves, 480.706 seconds. The Gradle result test passed.
- **Codex session ID:** `01a0d001-aeee-7cd2-b030-23c4ba86cabe`.
- **Protocol observation:** All 60 move responses satisfied the per-turn JSON schema and selected a listed legal command. Session creation and every resume succeeded.
- **Strategy observation:** The model produced a strong verbal strategy covering hidden-card reveals, safe foundation play, empty columns, stock memory, and short combinations.
- **Observed behavior:** Many stock turns. On the final board the model chose `quit` while three non-quit tableau moves remained: `move T3 8♥ T4`, `move T3 5♣ T5`, and `move T6 6♠ T7`. At least the last option was the exact inverse of an earlier `move T7 6♠ T6`; the legal-move generator does not filter repeats or visited states.
- **Interpretation:** Output formatting is not the first blocker. The remaining moves may all have been reversible rearrangements, so this run does not establish that quitting was strategically wrong. One random game is diagnostic only and says nothing reliable about win rate.
- **Decision for the next prompt version:** Do not change P0 based on this result alone. Continue with independent random games and use session transcripts plus terminal-state diagnostics to understand repeated failure patterns.

### Run `p0-sol-medium-100-20260925`: unassisted persistent-session baseline

- **Question:** Can `gpt-5.6-sol` formulate and execute a winning Klondike strategy without the engine's hand-authored strategy?
- **Prediction made before running:** P0 might produce no wins and require significant prompt refinement before the model could convert its verbal strategy into successful play.
- **Configuration:** `gpt-5.6-sol`, medium reasoning, Codex CLI 0.154.0, P0, one persistent session per game, engine guidance disabled, independent random shuffles, 1,000-move cap, and episode logging enabled.
- **Result:** 18/100 wins (18.00% ± 7.53% using the repository's normal-approximation notation; Wilson 95% interval approximately 11.7%-26.7%), 59.68 average moves, 474.557 average seconds per game, 47,455.715 total seconds, and a best win streak of 2.
- **Batch composition:** 10 completed games, then 1 and 9 completed games from two interrupted extension attempts, followed by a successful 80-game batch. The two games interrupted by infrastructure failures have no `EPISODE_SUMMARY` and are excluded.
- **Protocol observation:** All recorded commands belonged to their engine-generated legal-move sets. The final 80-game batch encountered eight transient failed CLI attempts; bounded retries recovered each one and the batch completed successfully.
- **Artifacts:** `engine/logs/game.log` contains model, reasoning, session ID, outcome, move count, and duration per completed game. `engine/logs/episode.log` contains exactly 100 completed summaries plus move-level state records; Codex session transcripts remain in the local Codex session store.
- **Interpretation:** The initial hypothesis was not supported. Sol won without receiving the engine's hand-authored strategy, showing that self-authored strategy plus persistent turn context is sufficient for meaningful play. The 18% estimate remains substantially below A* search and is dramatically slower, but it exceeds the historical 13% stateless API result in this sample. The experiments are not direct prompt or model comparisons.
- **Decision for the next prompt version:** Preserve P0 as the baseline. Analyse failures and terminal positions before proposing P1 or varying reasoning effort; do not retrospectively add strategy to this result.

### Run `p0-luna-medium-100-20260926`: unassisted persistent-session baseline

- **Question:** Can the lower-cost `gpt-5.6-luna` model formulate and execute a winning Klondike strategy under the same P0 persistent-session approach?
- **Configuration:** `gpt-5.6-luna`, medium reasoning, P0, one persistent session per game, engine guidance disabled, independent random shuffles, a 200-move cap, and episode logging enabled.
- **Result:** 5/100 wins (5.00% ± 4.27% using the repository's normal-approximation notation; Wilson 95% interval approximately 2.2%-11.2%), 130.78 average moves, 890.632 average seconds per game, 89,063.176 total seconds, and a best win streak of 1.
- **Winning games:** Games 5, 17, 46, 57, and 96 in the combined completed-game sequence.
- **Batch composition:** The original run completed games 1-97. Its game 98 stopped after 56 recorded steps during repeated Codex CLI failures and has no `EPISODE_SUMMARY`, so it is excluded. A separate three-game batch completed the sample; all three were losses.
- **Artifacts:** The original summaries and the three completion summaries are in `engine/logs/episode.log`; the completion batch has run ID `auto-37f359ad-68e5-4719-8f2c-e648b3d0cdcf`. Per-game results and Codex session IDs are in `engine/logs/game.log`.
- **Interpretation:** Luna can win with a self-authored strategy and no engine guidance, but its 5% result is materially below Sol's 18% result in these samples. The runs are not a fully controlled model comparison because Luna used a 200-move cap while Sol used a 1,000-move cap.
- **Decision for the next prompt version:** Preserve this result as the Luna P0 baseline. Apply the ten-game screening and episode-log rubric before committing Luna or another model to a larger follow-up run.

### Run `p0-copilot-haiku-smoke-20260927`: first GitHub Copilot CLI smoke game

- **Question:** Can an Anthropic model play through GitHub Copilot CLI using the same persistent-session, self-authored-strategy experiment?
- **Prediction made before running:** Copilot's coding-agent system prompt might reject inert game commands or fail to return structured output, so protocol behavior matters more than the outcome of this single deal.
- **Configuration:** `claude-haiku-4.5`, default reasoning, GitHub Copilot CLI, P0, one persistent session, empty temporary workspace, custom instructions/MCPs/tools disabled, engine guidance disabled, independent random shuffle, 200-move cap, and episode logging enabled.
- **Result:** 0/1 wins, 44 successful moves, 341.722 seconds, and a model-selected `quit`. The CLI reported 15.18 in its legacy-named `premiumRequests` usage field across the strategy and board turns.
- **Session ID:** `ac47524f-4ff5-4b4d-8cd1-c2d6fd9f9c2a`.
- **Protocol observation:** Session creation and every resume succeeded on the requested model. All 45 decision records selected an engine-listed command; no structured-output correction or transient process retry was needed.
- **Strategy observation:** Haiku described standard Klondike rules and prioritized hidden-card reveals, foundations, tableau sequencing, empty columns, and stock tracking without receiving the engine's hand-authored strategy.
- **Observed behavior:** The final position had 6 foundation cards and 20 face-down tableau cards. Haiku quit with only `turn` and `quit` listed, after repeatedly cycling the stock; this is a coherent terminal choice under the supplied interface but a strategically weak game.
- **Interpretation:** The Copilot CLI transport is viable for the experiment. One loss is diagnostic only and provides no win-rate estimate; the next decision should be based on the ten-game screening protocol and AI-credit cost.
- **Decision for the next prompt version:** Keep P0 unchanged. Use this implementation for a ten-game Haiku screen only after confirming available AI-credit headroom through GitHub's billing usage API.

### Runs `p0-jev-stateful-smoke-20260928` and `p1-jev-stateful-smoke-20260928`

- **Question:** Can Jev retain human-visible information across turns when the client resends structured observation history, and does P1 change its initial play?
- **Configuration:** `jev-1.13.0`, TypeSafe System One Choice API, independent random shuffles, full current board plus compact observations from every completed turn, engine guidance disabled, 200-move cap, and episode logging enabled. P0 supplied no strategic policy; P1 supplied corrected rules and eight ordered priorities.
- **Context-limit finding:** Sending every prior full board failed on turn 93 with `max_tokens_exceeded` at 32,768 input tokens. Compact historical observations retain foundation/tableau tops, tableau hidden counts, stock/waste state, feedback, and selected commands; both revised runs completed 200 decisions.
- **P0 result:** 0/1 wins in 71.478 seconds. Jev made 198 stock turns and two waste moves to foundations, reached two foundation cards, revealed no face-down tableau cards, and visited 18 unique states. Total reported usage was 3,238,750 input and 14,001 output tokens.
- **P1 result:** 0/1 wins in 69.480 seconds. Jev made 199 stock turns and one tableau move, revealed one face-down card, made no foundation progress, and visited 10 unique states. Total reported usage was 3,171,789 input and 7,872 output tokens.
- **Interpretation:** Stateful transport now works for a complete capped game, but neither smoke run played coherently. P1 changed behavior but did not produce an improvement in this sample. The games used different random deals and `n=1`, so they do not estimate a profile effect.
- **Decision for the next prompt version:** Inspect the option probabilities and repeated-turn states before revising P1. A likely refinement is to make progress and repetition explicit decision criteria rather than relying only on an ordered policy paragraph.

### Jev P1 refinement sequence: J-P1.1 through J-P1.5

- **Question:** Can structured option-level consequences turn Jev's legal but stock-heavy P1 play into measurable progress and wins?
- **J-P1.1:** Added one-step simulated hidden-card, foundation, and repeated-position consequences to every legal Choice option. The run reached 10 foundation cards and 7 hidden cards after 196 moves, but its verbose history exceeded Jev's 32,768-token input limit before turn 197 completed.
- **J-P1.2:** Compacted history and added numeric priority ranks. The game completed, but regressed to 194 stock turns, one foundation card, and 20 hidden cards. Numeric ranks were removed.
- **J-P1.3:** Combined compact history with J-P1.1's natural-language consequence descriptions. Two of three games made strong progress, each reaching 11 foundation cards and revealing 15 or 13 hidden cards; the third made 200 stock turns.
- **J-P1.4:** Added consecutive-stock-turn and turns-since-progress summaries. This eliminated the all-stock result in three games; the best ended with 10 foundation cards and only 3 hidden cards.
- **J-P1.5:** Kept every legal option but marked the best immediate hidden-card reveal, or foundation gain when no reveal existed, as `required_by_progress_rule`. The first three-game pilot produced no wins but included a position with 30 foundation cards and one hidden card.
- **Confirmation result:** A further five independent random games produced 2 wins (40.00% ± 42.94%) in 156 and 134 moves. The sample is deliberately small and cannot support a stable win-rate claim. The three losses remained stock-heavy, showing that P1.5 improves capable trajectories but does not eliminate failure modes.
- **Cost:** The refinement runs sent approximately 44.9 million input tokens after the initial stateful smokes, roughly $1.89 at the displayed $0.042 per million input tokens, before delayed usage reporting. This remained comfortably within the $10 experiment budget.
- **Decision:** Generalize the successful J-P1.5 experiment as the shared P1.5 policy in `GamePrompts`; TypeSafe supplies its computed move facts, while the rules and instructions remain reusable by other model players. Do not promote the 40% pilot to the root README result table; next use a larger fixed-contract random sample or investigate why some deals still enter stock loops.

### Run `p1.5-jev-detailed-100-20260928`: detailed-policy evaluation

- **Question:** Does the shared P1.5 policy turn Jev's typed legal choices into a measurable win rate over a 100-game random sample?
- **Configuration:** `jev-1.13.0`, TypeSafe System One Choice API, P1.5 detailed policy, compact observed-board history, engine guidance disabled, independent random shuffles, 200-move cap, and episode logging enabled.
- **Result:** 10/100 wins (10.00% ± 5.88% using the repository's normal-approximation notation; Wilson 95% interval approximately 5.5%-17.4%), 193.52 average moves, 65.582 average seconds per game, 6,558.203 total seconds, and a best win streak of 1.
- **Winning games:** Games 5, 12, 33, 48, 69, 71, 73, 75, 77, and 81 in the combined completed-game sequence.
- **Batch composition:** The first attempt completed 59 games before API credits ran out during game 60 after 185 moves. That partial game has no `EPISODE_SUMMARY` and is excluded. A second 41-game batch completed the 100-game sample.
- **Usage:** The 100 completed games reported 283,556,308 input tokens and 1,313,616 output tokens.
- **P0 control:** A separate ten-game P0 screen produced 0 wins; all ten games reached 200 moves, and 1,971 of 2,000 choices were stock turns. Its Wilson 95% interval is approximately 0.0%-27.8%, so the samples do not establish a precise causal effect, but their behavior strongly supports continued use of explicit strategy for Jev.
- **Artifacts:** Completed games 1-59 are under `engine/logs/typesafe-p1.5-100`; games 60-100 are under `engine/logs/typesafe-p1.5-remaining-41`; the P0 control is under `engine/logs/typesafe-p0-10`. Runtime logs remain ignored by Git.
- **Interpretation:** Jev can win a meaningful fraction of random deals when supplied the reusable P1.5 policy and option-level progress facts. The result is below Sol's 18% P0 result and above Luna's 5% P0 result, but those systems, prompts, and execution paths differ and should not be treated as a controlled model ranking.
- **Decision:** Promote the 100-game P1.5 result to the root README under Model-backed players. Preserve the ten-game P0 run as a diagnostic control in this notebook rather than a stable win-rate row.

## Prompt Refinement Guardrails

- Write down the predicted failure and expected improvement before editing the prompt.
- Change one conceptual element at a time when practical.
- Preserve old prompt versions and their results.
- Do not tune against one unusually easy or difficult random deal.
- Separate small exploratory pilots from larger random-sample evaluations.
- Treat model, reasoning effort, prompt, memory policy, and engine guidance as separate variables.
- Record unsuccessful prompt versions; they are part of the research result.

## Promotion Checklist

Move a result into the root `README.md` only when:

- The prompt version and experiment contract are frozen.
- The model was evaluated with the documented random-deal method and sample size.
- The sample size and confidence interval are reported.
- Aggregate results can be regenerated from machine-readable records.
- Session IDs or equivalent transcripts exist for qualitative review.
- Pilot and confirmatory runs are clearly distinguished.
- The README row names the exact model, reasoning effort, prompt version, and persistent-session behavior.

## Conclusions

The P0 baselines demonstrate that current Codex CLI models can turn a self-authored Klondike strategy into successful play without engine guidance. Sol won 18 of 100 independent random games and Luna won 5 of 100 while maintaining a persistent session for each deal. Jev's typed System One player won 0 of 10 games under P0 and 10 of 100 with the supplied P1.5 policy, showing that explicit domain guidance materially changes its observed behavior even though the independent samples do not isolate a causal effect. Model, prompt, context transport, and move cap all remain confounded across providers. A* remains stronger at 36.87% and much faster. Future comparisons should freeze one contract at a time and use the episode traces to classify failures before another large run.